In [200]:
import json
from pathlib import Path
from collections import defaultdict, Counter

In [201]:
data = {
    'task':'/Users/richardgurtsiev/Desktop/projects/vibe_code/workspaces/kib/data/вопросы/pool1.json',
    'gt':'/Users/richardgurtsiev/Desktop/projects/vibe_code/workspaces/kib/data/ответы/pool1.json'
}


with open(data['task'], 'r', encoding='utf-8') as f:
    tasks = json.load(f)

with open(data['gt'], 'r', encoding='utf-8') as f:
    gt = json.load(f)

In [202]:
def build_majority_dataset(gt: list[dict]) -> tuple[list[dict], list[dict]]:

    grouped = defaultdict(list)

    for item in gt:
        item_id = item.get('item_id')
        result = item.get('result')

        if item_id is None or not isinstance(result, dict):
            continue

        grouped[item_id].append(result)

    dataset = []
    ties = []

    for item_id, results in grouped.items():

        # Считаем частоту каждого уникального ответа
        result_counts = Counter(
            json.dumps(result, sort_keys=True, ensure_ascii=False)
            for result in results
        )

        max_votes = max(result_counts.values())

        # Все ответы с максимальной частотой
        winners = [
            json.loads(result)
            for result, count in result_counts.items()
            if count == max_votes
        ]

        record = {
            'item_id': item_id,
            'result': winners[0] if len(winners) == 1 else winners,
            'votes': max_votes,
            'total_votes': len(results),
        }

        if len(winners) == 1:
            dataset.append(record)
        else:
            ties.append(record)

    return dataset, ties

In [203]:
dataset, ties = build_majority_dataset(gt)

print(f'Всего уникальных заданий: {len(dataset) + len(ties)}')
print(f'Однозначное большинство: {len(dataset)}')
print(f'Ничьи: {len(ties)}')

Всего уникальных заданий: 50
Однозначное большинство: 36
Ничьи: 14


In [ ]:
def get_task_by_id(tasks: list[dict], task_id: str) -> dict | None:
    """
    Находит задание в tasks по task_id.

    Возвращает data задания или None, если задание не найдено.
    """
    for task in tasks:
        if task.get('data', {}).get('id') == task_id:
            return task['data']

    return None

In [ ]:
import json
from pathlib import Path
from collections import defaultdict, Counter


#### Чтение задания и ответов
data = {
    'task':'./вопросы/pool1.json',
    'gt':'./ответы/pool1.json'
}

with open(data['task'], 'r', encoding='utf-8') as f:
    tasks = json.load(f)

with open(data['gt'], 'r', encoding='utf-8') as f:
    gt = json.load(f)



#### Метод для отбора 

def build_majority_dataset(gt: list[dict]) -> tuple[list[dict], list[dict]]:

    grouped = defaultdict(list)

    for item in gt:
        item_id = item.get('item_id')
        result = item.get('result')

        if item_id is None or not isinstance(result, dict):
            continue

        grouped[item_id].append(result)

    dataset = []
    ties = []

    for item_id, results in grouped.items():

        # Считаем частоту каждого уникального ответа
        result_counts = Counter(
            json.dumps(result, sort_keys=True, ensure_ascii=False)
            for result in results
        )

        max_votes = max(result_counts.values())

        # Все ответы с максимальной частотой
        winners = [
            json.loads(result)
            for result, count in result_counts.items()
            if count == max_votes
        ]

        record = {
            'item_id': item_id,
            'result': winners[0] if len(winners) == 1 else winners,
            'votes': max_votes,
            'total_votes': len(results),
        }

        if len(winners) == 1:
            dataset.append(record)
        else:
            ties.append(record)

    return dataset, ties


dataset, ties = build_majority_dataset(gt)



#### Метод для поиска задания

def get_task_by_id(tasks: list[dict], task_id: str) -> dict | None:
    """
    Находит задание в tasks по task_id.

    Возвращает data задания или None, если задание не найдено.
    """
    for task in tasks:
        if task.get('data', {}).get('id') == task_id:
            return task['data']

    return None


#### Создание prompt 

prompt = """
Давай найдем лучший ответ среди ничьи. Я передам тебе само задания и ответы, а ты определишь среди несколько ответов что является наиболее вероятностной.


Задание:
```
{{task}}
```

Ответы:
```
{{ground_truth_str}}
```

""".strip()


task_id = ""
ground_truth = ""
ground_truths = []
ground_truths_str = []
prompts_task = []


for i in range(len(ties)):
    id = ties[i]['result'][0]['id']
    task_id = id

    for idx, item in enumerate(ties[i]['result']):
        resp = f"{idx + 1}. Ответ:" + " " + str(item)
        ground_truths.append(resp)

    ground_truth = "\n".join(ground_truths)
    ground_truths_str.append(ground_truth)


for ground_truth_str in ground_truths_str:
    task = get_task_by_id(tasks, task_id)
    
    prompt_task = prompt.format(
        ground_truth_str=ground_truth_str,
        task=task
    )
    prompts_task.append(prompt_task)



# #### Обращение по API к модели 
# ...


In [ ]:
def build_majority_dataset(gt: list[dict]) -> tuple[list[dict], list[dict]]:

    grouped = defaultdict(list)

    for item in gt:
        item_id = item.get('item_id')
        result = item.get('result')

        if item_id is None or not isinstance(result, dict):
            continue

        grouped[item_id].append(result)

    dataset = []
    ties = []

    for item_id, results in grouped.items():

        # Считаем частоту каждого уникального ответа
        result_counts = Counter(
            json.dumps(result, sort_keys=True, ensure_ascii=False)
            for result in results
        )

        max_votes = max(result_counts.values())

        # Все ответы с максимальной частотой
        winners = [
            json.loads(result)
            for result, count in result_counts.items()
            if count == max_votes
        ]

        record = {
            'item_id': item_id,
            'result': winners[0] if len(winners) == 1 else winners,
            'votes': max_votes,
            'total_votes': len(results),
        }

        if len(winners) == 1:
            dataset.append(record)
        else:
            ties.append(record)

    return dataset, ties


data = {
    'task':'/Users/richardgurtsiev/Desktop/projects/vibe_code/workspaces/kib/data/вопросы/pool1.json',
    'gt':'/Users/richardgurtsiev/Desktop/projects/vibe_code/workspaces/kib/data/ответы/pool1.json'
}


with open(data['task'], 'r', encoding='utf-8') as f:
    tasks = json.load(f)

with open(data['gt'], 'r', encoding='utf-8') as f:
    gt = json.load(f)

In [231]:
import json

from collections import defaultdict, Counter


# ============================================================
# 1. ФОРМИРОВАНИЕ DATASET И TIES
# ============================================================

def build_majority_dataset(
    gt: list[dict]
) -> tuple[list[dict], list[dict]]:

    grouped = defaultdict(list)
    invalid_records = 0

    for item in gt:

        item_id = item.get('item_id')
        result = item.get('result')

        if item_id is None or not isinstance(result, dict):
            invalid_records += 1
            continue

        grouped[item_id].append(result)

    dataset = []
    ties = []

    for item_id, results in grouped.items():

        result_counts = Counter(
            json.dumps(
                result,
                sort_keys=True,
                ensure_ascii=False
            )
            for result in results
        )

        max_votes = max(result_counts.values())

        winners = [
            json.loads(result)
            for result, count in result_counts.items()
            if count == max_votes
        ]

        record = {
            'item_id': item_id,
            'result': winners[0] if len(winners) == 1 else winners,
            'votes': max_votes,
            'total_votes': len(results),
        }

        if len(winners) == 1:
            dataset.append(record)
        else:
            ties.append(record)

    # if invalid_records:
    #     print(f"Некорректных записей пропущено: {invalid_records}")

    return dataset, ties


# ============================================================
# 2. ЗАГРУЗКА ДАННЫХ
# ============================================================

data = {
    'task': '/Users/richardgurtsiev/Desktop/projects/vibe_code/workspaces/kib/data/вопросы/pool6.json',
    'gt': '/Users/richardgurtsiev/Desktop/projects/vibe_code/workspaces/kib/data/ответы/pool6.json'
}

with open(data['task'], 'r', encoding='utf-8') as f:
    tasks = json.load(f)

with open(data['gt'], 'r', encoding='utf-8') as f:
    gt = json.load(f)


# ============================================================
# 3. ФОРМИРОВАНИЕ DATASET И TIES
# ============================================================

dataset, ties = build_majority_dataset(gt)

print(f"Всего заданий: {len(dataset) + len(ties)}")
print(f"Без ничьих: {len(dataset)}")
print(f"С ничьими: {len(ties)}")


# ============================================================
# 4. ИНДЕКС ИСХОДНЫХ ЗАДАНИЙ
# ============================================================

tasks_by_id = {
    str(task['data']['id']): task['data']
    for task in tasks
    if isinstance(task.get('data'), dict)
    and task['data'].get('id') is not None
}


# ============================================================
# 5. ФОРМИРОВАНИЕ PROMPTS_TASK
# ============================================================

prompts_task = []

for tie in ties:

    results = tie['result']

    if not results:
        continue

    # Получаем ID задания из вариантов ответов
    task_ids = {
        str(result.get('id'))
        for result in results
        if result.get('id') is not None
    }

    # Проверяем, что у всех вариантов есть одинаковый ID
    if len(task_ids) != 1 or any(
        result.get('id') is None for result in results
    ):
        print(
            f"Некорректные ID: "
            f"item_id={tie['item_id']}, ids={task_ids}"
        )
        continue

    task_id = next(iter(task_ids))

    # Находим исходную карточку
    task = tasks_by_id.get(task_id)

    if task is None:
        print(f"Задание не найдено: {task_id}")
        continue

    # Формируем варианты ответов
    ground_truths = []

    for idx, result in enumerate(results, start=1):

        resp = (
            f"{idx}. Ответ: "
            f"{json.dumps(result, ensure_ascii=False)}"
        )

        ground_truths.append(resp)

    ground_truth_str = "\n".join(ground_truths)

    # Сохраняем задание
    prompts_task.append({
        "task_id": task_id,
        "item_id": tie["item_id"],
        "task": task,
        "results": results,
        "ground_truth_str": ground_truth_str,
    })


# ============================================================
# 6. СТАТИСТИКА
# ============================================================

print(f"\nПодготовлено заданий для модели: {len(prompts_task)}")
print(f"Не найдено или пропущено: {len(ties) - len(prompts_task)}")

Всего заданий: 150
Без ничьих: 118
С ничьими: 32

Подготовлено заданий для модели: 32
Не найдено или пропущено: 0


In [150]:
# for idx, item in enumerate(prompts_task[12+12+12:12+12+12+12], start=1):
#     example = f"Пример {idx}" + "\n " + str(item)
#     print(example)
#     print('\n\n')

## Pool 1

In [205]:
answers = [
    {
        "task_id": "071026_5",
        "result": {
            "agent_code": None,
            "id": "071026_5",
            "tool_needed": "no",
            "no_tool_reason": "clarification_needed"
        }
    },
    {
        "task_id": "071026_10",
        "result": {
            "agent_code": None,
            "id": "071026_10",
            "tool_needed": "no",
            "no_tool_reason": "missing_data"
        }
    },
    {
        "task_id": "071026_11",
        "result": {
            "agent_code": "plan_execute",
            "id": "071026_11",
            "tool_needed": "yes"
        }
    },
    {
        "task_id": "071026_17",
        "result": {
            "agent_code": None,
            "id": "071026_17",
            "tool_needed": "no",
            "no_tool_reason": "no_suitable_tool"
        }
    },
    {
        "task_id": "071026_20",
        "result": {
            "agent_code": "get_contest_methodology_tool",
            "id": "071026_20",
            "tool_needed": "yes"
        }
    },
    {
        "task_id": "071026_21",
        "result": {
            "agent_code": "get_tribe",
            "id": "071026_21",
            "tool_needed": "yes"
        }
    },
    {
        "task_id": "071026_23",
        "result": {
            "agent_code": None,
            "id": "071026_23",
            "tool_needed": "no",
            "no_tool_reason": "no_suitable_tool"
        }
    },
    {
        "task_id": "071026_24",
        "result": {
            "agent_code": None,
            "id": "071026_24",
            "tool_needed": "no",
            "no_tool_reason": "no_tool_required"
        }
    },
    {
        "task_id": "071026_25",
        "result": {
            "agent_code": None,
            "id": "071026_25",
            "tool_needed": "no",
            "no_tool_reason": "clarification_needed"
        }
    },
    {
        "task_id": "071026_26",
        "result": {
            "agent_code": None,
            "id": "071026_26",
            "tool_needed": "no",
            "no_tool_reason": "no_suitable_tool"
        }
    },
    {
        "task_id": "071026_27",
        "result": {
            "agent_code": "get_contest_methodology_tool",
            "id": "071026_27",
            "tool_needed": "yes"
        }
    },
    {
        "task_id": "071026_28",
        "result": {
            "agent_code": "plan_execute",
            "id": "071026_28",
            "tool_needed": "yes"
        }
    },
    {
        "task_id": "071026_32",
        "result": {
            "agent_code": None,
            "id": "071026_32",
            "tool_needed": "no",
            "no_tool_reason": "clarification_needed"
        }
    },
    {
        "task_id": "071026_41",
        "result": {
            "agent_code": "plan_execute",
            "id": "071026_41",
            "tool_needed": "yes"
        }
    }
]


## Pool 2

In [217]:
answers = [
    {
        'task_id': '071026_56',
        'result': {
            'agent_code': None,
            'id': '071026_56',
            'tool_needed': 'no',
            'no_tool_reason': 'no_suitable_tool'
        }
    },
    {
        'task_id': '071026_62',
        'result': {
            'agent_code': None,
            'id': '071026_62',
            'tool_needed': 'no',
            'no_tool_reason': 'clarification_needed'
        }
    },
    {
        'task_id': '071026_66',
        'result': {
            'agent_code': None,
            'id': '071026_66',
            'tool_needed': 'no',
            'no_tool_reason': 'clarification_needed'
        }
    },
    {
        'task_id': '071026_74',
        'result': {
            'agent_code': None,
            'id': '071026_74',
            'tool_needed': 'no',
            'no_tool_reason': 'no_suitable_tool'
        }
    },
    {
        'task_id': '071026_75',
        'result': {
            'agent_code': 'get_tribe',
            'id': '071026_75',
            'tool_needed': 'yes'
        }
    },
    {
        'task_id': '071026_78',
        'result': {
            'agent_code': None,
            'id': '071026_78',
            'tool_needed': 'no',
            'no_tool_reason': 'no_suitable_tool'
        }
    },
    {
        'task_id': '071026_81',
        'result': {
            'agent_code': None,
            'id': '071026_81',
            'tool_needed': 'no',
            'no_tool_reason': 'no_tool_required'
        }
    },
    {
        'task_id': '071026_80',
        'result': {
            'agent_code': 'what_can_i_do',
            'id': '071026_80',
            'tool_needed': 'yes'
        }
    },
    {
        'task_id': '071026_85',
        'result': {
            'agent_code': None,
            'id': '071026_85',
            'tool_needed': 'no',
            'no_tool_reason': 'no_suitable_tool'
        }
    },
    {
        'task_id': '071026_84',
        'result': {
            'agent_code': 'plan_execute',
            'id': '071026_84',
            'tool_needed': 'yes'
        }
    },
    {
        'task_id': '071026_94',
        'result': {
            'agent_code': None,
            'id': '071026_94',
            'tool_needed': 'no',
            'no_tool_reason': 'missing_data'
        }
    },
    {
        'task_id': '071026_95',
        'result': {
            'agent_code': None,
            'id': '071026_95',
            'tool_needed': 'no',
            'no_tool_reason': 'no_suitable_tool'
        }
    },
    {
        'task_id': '071026_98',
        'result': {
            'agent_code': None,
            'id': '071026_98',
            'tool_needed': 'no',
            'no_tool_reason': 'clarification_needed'
        }
    },
    {
        'task_id': '071026_106',
        'result': {
            'agent_code': None,
            'id': '071026_106',
            'tool_needed': 'no',
            'no_tool_reason': 'clarification_needed'
        }
    },
    {
        'task_id': '071026_112',
        'result': {
            'agent_code': None,
            'id': '071026_112',
            'tool_needed': 'no',
            'no_tool_reason': 'no_suitable_tool'
        }
    },
    {
        'task_id': '071026_121',
        'result': {
            'agent_code': None,
            'id': '071026_121',
            'tool_needed': 'no',
            'no_tool_reason': 'no_suitable_tool'
        }
    },
    {
        'task_id': '071026_127',
        'result': {
            'agent_code': None,
            'id': '071026_127',
            'tool_needed': 'no',
            'no_tool_reason': 'missing_data'
        }
    },
    {
        'task_id': '071026_133',
        'result': {
            'agent_code': 'plan_execute',
            'id': '071026_133',
            'tool_needed': 'yes'
        }
    },
    {
        'task_id': '071026_137',
        'result': {
            'agent_code': 'what_can_i_do',
            'id': '071026_137',
            'tool_needed': 'yes'
        }
    },
    {
        'task_id': '071026_141',
        'result': {
            'agent_code': None,
            'id': '071026_141',
            'tool_needed': 'no',
            'no_tool_reason': 'no_suitable_tool'
        }
    },
    {
        'task_id': '071026_149',
        'result': {
            'agent_code': None,
            'id': '071026_149',
            'tool_needed': 'no',
            'no_tool_reason': 'clarification_needed'
        }
    },
    {
        'task_id': '071026_148',
        'result': {
            'agent_code': 'kpk_tool',
            'id': '071026_148',
            'tool_needed': 'yes'
        }
    },
    {
        'task_id': '071026_153',
        'result': {
            'agent_code': None,
            'id': '071026_153',
            'tool_needed': 'no',
            'no_tool_reason': 'no_suitable_tool'
        }
    },
    {
        'task_id': '071026_171',
        'result': {
            'agent_code': 'get_contest_methodology_tool',
            'id': '071026_171',
            'tool_needed': 'yes'
        }
    },
    {
        'task_id': '071026_173',
        'result': {
            'agent_code': None,
            'id': '071026_173',
            'tool_needed': 'no',
            'no_tool_reason': 'clarification_needed'
        }
    },
    {
        'task_id': '071026_176',
        'result': {
            'agent_code': 'get_bank_improvement_tool',
            'id': '071026_176',
            'tool_needed': 'yes'
        }
    },
    {
        'task_id': '071026_180',
        'result': {
            'agent_code': None,
            'id': '071026_180',
            'tool_needed': 'no',
            'no_tool_reason': 'no_suitable_tool'
        }
    },
    {
        'task_id': '071026_185',
        'result': {
            'agent_code': None,
            'id': '071026_185',
            'tool_needed': 'no',
            'no_tool_reason': 'no_suitable_tool'
        }
    },
    {
        'task_id': '071026_189',
        'result': {
            'agent_code': None,
            'id': '071026_189',
            'tool_needed': 'no',
            'no_tool_reason': 'no_suitable_tool'
        }
    },
    {
        'task_id': '071026_187',
        'result': {
            'agent_code': 'get_contest_methodology_tool',
            'id': '071026_187',
            'tool_needed': 'yes'
        }
    },
    {
        'task_id': '071026_195',
        'result': {
            'agent_code': None,
            'id': '071026_195',
            'tool_needed': 'no',
            'no_tool_reason': 'unclear_request'
        }
    },
    {
        'task_id': '071026_201',
        'result': {
            'agent_code': None,
            'id': '071026_201',
            'tool_needed': 'no',
            'no_tool_reason': 'no_suitable_tool'
        }
    },
    {
        'task_id': '071026_203',
        'result': {
            'agent_code': 'plan_execute',
            'id': '071026_203',
            'tool_needed': 'yes'
        }
    },
    {
        'task_id': '071026_209',
        'result': {
            'agent_code': 'get_contest_methodology_tool',
            'id': '071026_209',
            'tool_needed': 'yes'
        }
    },
    {
        'task_id': '071026_219',
        'result': {
            'agent_code': 'get_contest_methodology_tool',
            'id': '071026_219',
            'tool_needed': 'yes'
        }
    },
    {
        'task_id': '071026_226',
        'result': {
            'agent_code': None,
            'id': '071026_226',
            'tool_needed': 'no',
            'no_tool_reason': 'clarification_needed'
        }
    },
    {
        'task_id': '071026_238',
        'result': {
            'agent_code': None,
            'id': '071026_238',
            'tool_needed': 'no',
            'no_tool_reason': 'no_suitable_tool'
        }
    },
    {
        'task_id': '071026_239',
        'result': {
            'agent_code': None,
            'id': '071026_239',
            'tool_needed': 'no',
            'no_tool_reason': 'clarification_needed'
        }
    },
    {
        'task_id': '071026_242',
        'result': {
            'agent_code': None,
            'id': '071026_242',
            'tool_needed': 'no',
            'no_tool_reason': 'clarification_needed'
        }
    },
    {
        'task_id': '071026_247',
        'result': {
            'agent_code': 'get_contest_methodology_tool',
            'id': '071026_247',
            'tool_needed': 'yes'
        }
    }
]

len(answers)

40

## Pools 3

In [219]:
answers = [
  {
    "task_id": "071026_252",
    "result": {
      "agent_code": None,
      "id": "071026_252",
      "tool_needed": "no",
      "no_tool_reason": "clarification_needed"
    }
  },
  {
    "task_id": "071026_257",
    "result": {
      "agent_code": None,
      "id": "071026_257",
      "tool_needed": "no",
      "no_tool_reason": "no_suitable_tool"
    }
  },
  {
    "task_id": "071026_259",
    "result": {
      "agent_code": "plan_execute",
      "id": "071026_259",
      "tool_needed": "yes"
    }
  },
  {
    "task_id": "071026_266",
    "result": {
      "agent_code": None,
      "id": "071026_266",
      "tool_needed": "no",
      "no_tool_reason": "no_suitable_tool"
    }
  },
  {
    "task_id": "071026_273",
    "result": {
      "agent_code": "get_contest_methodology_tool",
      "id": "071026_273",
      "tool_needed": "yes"
    }
  },
  {
    "task_id": "071026_288",
    "result": {
      "agent_code": "get_contest_methodology_tool",
      "id": "071026_288",
      "tool_needed": "yes"
    }
  },
  {
    "task_id": "071026_290",
    "result": {
      "agent_code": "kpk_tool",
      "id": "071026_290",
      "tool_needed": "yes"
    }
  },
  {
    "task_id": "071026_296",
    "result": {
      "agent_code": "get_contest_methodology_tool",
      "id": "071026_296",
      "tool_needed": "yes"
    }
  },
  {
    "task_id": "071026_295",
    "result": {
      "agent_code": None,
      "id": "071026_295",
      "tool_needed": "no",
      "no_tool_reason": "no_suitable_tool"
    }
  },
  {
    "task_id": "071026_298",
    "result": {
      "agent_code": None,
      "id": "071026_298",
      "tool_needed": "no",
      "no_tool_reason": "no_suitable_tool"
    }
  },
  {
    "task_id": "071026_301",
    "result": {
      "agent_code": "get_bank_metric_tool",
      "id": "071026_301",
      "tool_needed": "yes"
    }
  },
  {
    "task_id": "071026_303",
    "result": {
      "agent_code": "planair_fa",
      "id": "071026_303",
      "tool_needed": "yes"
    }
  },
  {
    "task_id": "071026_308",
    "result": {
      "agent_code": None,
      "id": "071026_308",
      "tool_needed": "no",
      "no_tool_reason": "no_suitable_tool"
    }
  },
  {
    "task_id": "071026_315",
    "result": {
      "agent_code": None,
      "id": "071026_315",
      "tool_needed": "no",
      "no_tool_reason": "no_suitable_tool"
    }
  },
  {
    "task_id": "071026_316",
    "result": {
      "agent_code": "get_contest_methodology_tool",
      "id": "071026_316",
      "tool_needed": "yes"
    }
  },
  {
    "task_id": "071026_319",
    "result": {
      "agent_code": None,
      "id": "071026_319",
      "tool_needed": "no",
      "no_tool_reason": "clarification_needed"
    }
  },
  {
    "task_id": "071026_322",
    "result": {
      "agent_code": None,
      "id": "071026_322",
      "tool_needed": "no",
      "no_tool_reason": "no_suitable_tool"
    }
  },
  {
    "task_id": "071026_327",
    "result": {
      "agent_code": "plan_execute",
      "id": "071026_327",
      "tool_needed": "yes"
    }
  },
  {
    "task_id": "071026_329",
    "result": {
      "agent_code": "plan_execute",
      "id": "071026_329",
      "tool_needed": "yes"
    }
  },
  {
    "task_id": "071026_330",
    "result": {
      "agent_code": "get_contest_methodology_tool",
      "id": "071026_330",
      "tool_needed": "yes"
    }
  },
  {
    "task_id": "071026_333",
    "result": {
      "agent_code": None,
      "id": "071026_333",
      "tool_needed": "no",
      "no_tool_reason": "unclear_request"
    }
  },
  {
    "task_id": "071026_341",
    "result": {
      "agent_code": "plan_execute",
      "id": "071026_341",
      "tool_needed": "yes"
    }
  },
  {
    "task_id": "071026_345",
    "result": {
      "agent_code": None,
      "id": "071026_345",
      "tool_needed": "no",
      "no_tool_reason": "no_suitable_tool"
    }
  },
  {
    "task_id": "071026_350",
    "result": {
      "agent_code": None,
      "id": "071026_350",
      "tool_needed": "no",
      "no_tool_reason": "no_suitable_tool"
    }
  },
  {
    "task_id": "071026_354",
    "result": {
      "agent_code": "get_contest_methodology_tool",
      "id": "071026_354",
      "tool_needed": "yes"
    }
  },
  {
    "task_id": "071026_357",
    "result": {
      "agent_code": None,
      "id": "071026_357",
      "tool_needed": "no",
      "no_tool_reason": "no_suitable_tool"
    }
  },
  {
    "task_id": "071026_367",
    "result": {
      "agent_code": "get_bank_metric_tool",
      "id": "071026_367",
      "tool_needed": "yes"
    }
  },
  {
    "task_id": "071026_374",
    "result": {
      "agent_code": None,
      "id": "071026_374",
      "tool_needed": "no",
      "no_tool_reason": "missing_data"
    }
  },
  {
    "task_id": "071026_380",
    "result": {
      "agent_code": "get_kpi_rgs",
      "id": "071026_380",
      "tool_needed": "yes"
    }
  },
  {
    "task_id": "071026_381",
    "result": {
      "agent_code": "get_contest_methodology_tool",
      "id": "071026_381",
      "tool_needed": "yes"
    }
  },
  {
    "task_id": "071026_387",
    "result": {
      "agent_code": None,
      "id": "071026_387",
      "tool_needed": "no",
      "no_tool_reason": "no_suitable_tool"
    }
  },
  {
    "task_id": "071026_390",
    "result": {
      "agent_code": None,
      "id": "071026_390",
      "tool_needed": "no",
      "no_tool_reason": "clarification_needed"
    }
  },
  {
    "task_id": "071026_389",
    "result": {
      "agent_code": None,
      "id": "071026_389",
      "tool_needed": "no",
      "no_tool_reason": "unclear_request"
    }
  },
  {
    "task_id": "071026_395",
    "result": {
      "agent_code": "get_contest_methodology_tool",
      "id": "071026_395",
      "tool_needed": "yes"
    }
  },
  {
    "task_id": "071026_396",
    "result": {
      "agent_code": None,
      "id": "071026_396",
      "tool_needed": "no",
      "no_tool_reason": "no_suitable_tool"
    }
  },
  {
    "task_id": "071026_401",
    "result": {
      "agent_code": "get_contest_methodology_tool",
      "id": "071026_401",
      "tool_needed": "yes"
    }
  },
  {
    "task_id": "071026_400",
    "result": {
      "agent_code": "get_contest_methodology_tool",
      "id": "071026_400",
      "tool_needed": "yes"
    }
  },
  {
    "task_id": "071026_407",
    "result": {
      "agent_code": None,
      "id": "071026_407",
      "no_tool_reason": "missing_data",
      "tool_needed": "no"
    }
  },
  {
    "task_id": "071026_411",
    "result": {
      "agent_code": None,
      "id": "071026_411",
      "no_tool_reason": "no_suitable_tool",
      "tool_needed": "no"
    }
  },
  {
    "task_id": "071026_412",
    "result": {
      "agent_code": "get_contest_result",
      "id": "071026_412",
      "tool_needed": "yes"
    }
  },
  {
    "task_id": "071026_413",
    "result": {
      "agent_code": "get_contest_methodology_tool",
      "id": "071026_413",
      "tool_needed": "yes"
    }
  },
  {
    "task_id": "071026_416",
    "result": {
      "agent_code": None,
      "id": "071026_416",
      "no_tool_reason": "no_suitable_tool",
      "tool_needed": "no"
    }
  },
  {
    "task_id": "071026_425",
    "result": {
      "agent_code": None,
      "id": "071026_425",
      "no_tool_reason": "clarification_needed",
      "tool_needed": "no"
    }
  },
  {
    "task_id": "071026_427",
    "result": {
      "agent_code": "get_contest_methodology_tool",
      "id": "071026_427",
      "tool_needed": "yes"
    }
  },
  {
    "task_id": "071026_431",
    "result": {
      "agent_code": None,
      "id": "071026_431",
      "no_tool_reason": "no_suitable_tool",
      "tool_needed": "no"
    }
  },
  {
    "task_id": "071026_432",
    "result": {
      "agent_code": "get_contest_methodology_tool",
      "id": "071026_432",
      "tool_needed": "yes"
    }
  },
  {
    "task_id": "071026_436",
    "result": {
      "agent_code": "get_tribe",
      "id": "071026_436",
      "tool_needed": "yes"
    }
  },
  {
    "task_id": "071026_437",
    "result": {
      "agent_code": "get_contest_methodology_tool",
      "id": "071026_437",
      "tool_needed": "yes"
    }
  },
{
  "task_id": "071026_438",
  "result": {
    "agent_code": None,
    "id": "071026_438",
    "tool_needed": "no",
    "no_tool_reason": "no_suitable_tool"
  }
}
]
len(answers)

49

## Pool4

In [222]:
answers = [
  {
    "task_id": "071026_451",
    "result": {
      "agent_code": None,
      "id": "071026_451",
      "tool_needed": "no",
      "no_tool_reason": "clarification_needed"
    }
  },
  {
    "task_id": "071026_453",
    "result": {
      "agent_code": None,
      "id": "071026_453",
      "tool_needed": "no",
      "no_tool_reason": "unclear_request"
    }
  },
  {
    "task_id": "071026_455",
    "result": {
      "agent_code": None,
      "id": "071026_455",
      "tool_needed": "no",
      "no_tool_reason": "clarification_needed"
    }
  },
  {
    "task_id": "071026_459",
    "result": {
      "agent_code": None,
      "id": "071026_459",
      "tool_needed": "no",
      "no_tool_reason": "clarification_needed"
    }
  },
  {
    "task_id": "071026_469",
    "result": {
      "agent_code": "get_contest_methodology_tool",
      "id": "071026_469",
      "tool_needed": "yes"
    }
  },
  {
    "task_id": "071026_474",
    "result": {
      "agent_code": None,
      "id": "071026_474",
      "tool_needed": "no",
      "no_tool_reason": "clarification_needed"
    }
  },
  {
    "task_id": "071026_477",
    "result": {
      "agent_code": "plan_execute",
      "id": "071026_477",
      "tool_needed": "yes"
    }
  },
  {
    "task_id": "071026_480",
    "result": {
      "agent_code": "get_contest_methodology_tool",
      "id": "071026_480",
      "tool_needed": "yes"
    }
  },
  {
    "task_id": "071026_481",
    "result": {
      "agent_code": "get_contest_methodology_tool",
      "id": "071026_481",
      "tool_needed": "yes"
    }
  },
  {
    "task_id": "071026_482",
    "result": {
      "agent_code": None,
      "id": "071026_482",
      "tool_needed": "no",
      "no_tool_reason": "no_suitable_tool"
    }
  },
  {
    "task_id": "071026_483",
    "result": {
      "agent_code": None,
      "id": "071026_483",
      "tool_needed": "no",
      "no_tool_reason": "clarification_needed"
    }
  },
  {
    "task_id": "071026_485",
    "result": {
      "agent_code": "plan_execute",
      "id": "071026_485",
      "tool_needed": "yes"
    }
  },
  {
    "task_id": "071026_488",
    "result": {
      "agent_code": "get_contest_methodology_tool",
      "id": "071026_488",
      "tool_needed": "yes"
    }
  },
  {
    "task_id": "071026_494",
    "result": {
      "agent_code": None,
      "id": "071026_494",
      "tool_needed": "no",
      "no_tool_reason": "clarification_needed"
    }
  },
  {
    "task_id": "071026_495",
    "result": {
      "agent_code": "get_contest_methodology_tool",
      "id": "071026_495",
      "tool_needed": "yes"
    }
  },
  {
    "task_id": "071026_498",
    "result": {
      "agent_code": "get_bank_improvement_tool",
      "id": "071026_498",
      "tool_needed": "yes"
    }
  },
  {
    "task_id": "071026_503",
    "result": {
      "agent_code": None,
      "id": "071026_503",
      "tool_needed": "no",
      "no_tool_reason": "clarification_needed"
    }
  },
  {
    "task_id": "071026_505",
    "result": {
      "agent_code": None,
      "id": "071026_505",
      "tool_needed": "no",
      "no_tool_reason": "clarification_needed"
    }
  },
  {
    "task_id": "071026_512",
    "result": {
      "agent_code": "kpk_tool",
      "id": "071026_512",
      "tool_needed": "yes"
    }
  },
  {
    "task_id": "071026_514",
    "result": {
      "agent_code": None,
      "id": "071026_514",
      "tool_needed": "no",
      "no_tool_reason": "other"
    }
  },
  {
    "task_id": "071026_517",
    "result": {
      "agent_code": "plan_execute",
      "id": "071026_517",
      "tool_needed": "yes"
    }
  },
  {
    "task_id": "071026_525",
    "result": {
      "agent_code": None,
      "id": "071026_525",
      "tool_needed": "no",
      "no_tool_reason": "clarification_needed"
    }
  },
  {
    "task_id": "071026_524",
    "result": {
      "agent_code": "get_contest_methodology_tool",
      "id": "071026_524",
      "tool_needed": "yes"
    }
  },
  {
    "task_id": "071026_529",
    "result": {
      "agent_code": "get_bank_metric_tool",
      "id": "071026_529",
      "tool_needed": "yes"
    }
  },
  {
    "task_id": "071026_540",
    "result": {
      "agent_code": None,
      "id": "071026_540",
      "tool_needed": "no",
      "no_tool_reason": "no_suitable_tool"
    }
  },
  {
    "task_id": "071026_545",
    "result": {
      "agent_code": "plan_execute",
      "id": "071026_545",
      "tool_needed": "yes"
    }
  },
  {
    "task_id": "071026_550",
    "result": {
      "agent_code": "kpk_tool",
      "id": "071026_550",
      "tool_needed": "yes"
    }
  },
  {
    "task_id": "071026_553",
    "result": {
      "agent_code": "get_contest_methodology_tool",
      "id": "071026_553",
      "tool_needed": "yes"
    }
  },
  {
    "task_id": "071026_560",
    "result": {
      "agent_code": None,
      "id": "071026_560",
      "tool_needed": "no",
      "no_tool_reason": "clarification_needed"
    }
  },
  {
    "task_id": "071026_563",
    "result": {
      "agent_code": "get_contest_methodology_tool",
      "id": "071026_563",
      "tool_needed": "yes"
    }
  },
  {
    "task_id": "071026_570",
    "result": {
      "agent_code": "get_contest_result",
      "id": "071026_570",
      "tool_needed": "yes"
    }
  },
  {
    "task_id": "071026_574",
    "result": {
      "agent_code": "plan_execute",
      "id": "071026_574",
      "tool_needed": "yes"
    }
  },
  {
    "task_id": "071026_576",
    "result": {
      "agent_code": "plan_execute",
      "id": "071026_576",
      "tool_needed": "yes"
    }
  },
  {
    "task_id": "071026_582",
    "result": {
      "agent_code": None,
      "id": "071026_582",
      "tool_needed": "no",
      "no_tool_reason": "clarification_needed"
    }
  },
  {
    "task_id": "071026_588",
    "result": {
      "agent_code": None,
      "id": "071026_588",
      "tool_needed": "no",
      "no_tool_reason": "clarification_needed"
    }
  },
  {
    "task_id": "071026_592",
    "result": {
      "agent_code": "planair_fa",
      "id": "071026_592",
      "tool_needed": "yes"
    }
  },
  {
    "task_id": "071026_593",
    "result": {
      "agent_code": "get_contest_methodology_tool",
      "id": "071026_593",
      "tool_needed": "yes"
    }
  },
  {
    "task_id": "071026_594",
    "result": {
      "agent_code": None,
      "id": "071026_594",
      "tool_needed": "no",
      "no_tool_reason": "clarification_needed"
    }
  },
  {
    "task_id": "071026_596",
    "result": {
      "agent_code": None,
      "id": "071026_596",
      "tool_needed": "no",
      "no_tool_reason": "missing_data"
    }
  },
  {
    "task_id": "071026_615",
    "result": {
      "agent_code": "kpk_tool",
      "id": "071026_615",
      "tool_needed": "yes"
    }
  },
  {
    "task_id": "071026_624",
    "result": {
      "agent_code": "get_contest_methodology_tool",
      "id": "071026_624",
      "tool_needed": "yes"
    }
  },
  {
    "task_id": "071026_625",
    "result": {
      "agent_code": None,
      "id": "071026_625",
      "tool_needed": "no",
      "no_tool_reason": "no_suitable_tool"
    }
  },
  {
    "task_id": "071026_628",
    "result": {
      "agent_code": None,
      "id": "071026_628",
      "tool_needed": "no",
      "no_tool_reason": "clarification_needed"
    }
  },
  {
    "task_id": "071026_637",
    "result": {
      "agent_code": None,
      "id": "071026_637",
      "tool_needed": "no",
      "no_tool_reason": "no_suitable_tool"
    }
  },
  {
    "task_id": "071026_644",
    "result": {
      "agent_code": "get_contest_methodology_tool",
      "id": "071026_644",
      "tool_needed": "yes"
    }
  },
  {
    "task_id": "071026_646",
    "result": {
      "agent_code": None,
      "id": "071026_646",
      "tool_needed": "no",
      "no_tool_reason": "no_suitable_tool"
    }
  },
  {
    "task_id": "071026_649",
    "result": {
      "agent_code": "get_contest_result",
      "id": "071026_649",
      "tool_needed": "yes"
    }
  }
]

len(answers)

47

## Pool5

In [227]:
answers = [
  {
    "task_id": "071026_654",
    "result": {
      "agent_code": None,
      "id": "071026_654",
      "tool_needed": "no",
      "no_tool_reason": "no_suitable_tool"
    }
  },
  {
    "task_id": "071026_656",
    "result": {
      "agent_code": "plan_execute",
      "id": "071026_656",
      "tool_needed": "yes"
    }
  },
  {
    "task_id": "071026_659",
    "result": {
      "agent_code": None,
      "id": "071026_659",
      "tool_needed": "no",
      "no_tool_reason": "missing_data"
    }
  },
  {
    "task_id": "071026_660",
    "result": {
      "agent_code": "get_contest_methodology_tool",
      "id": "071026_660",
      "tool_needed": "yes"
    }
  },
  {
    "task_id": "071026_661",
    "result": {
      "agent_code": None,
      "id": "071026_661",
      "tool_needed": "no",
      "no_tool_reason": "no_suitable_tool"
    }
  },
  {
    "task_id": "071026_668",
    "result": {
      "agent_code": None,
      "id": "071026_668",
      "tool_needed": "no",
      "no_tool_reason": "no_suitable_tool"
    }
  },
  {
    "task_id": "071026_670",
    "result": {
      "agent_code": "kpk_tool",
      "id": "071026_670",
      "tool_needed": "yes"
    }
  },
  {
    "task_id": "071026_674",
    "result": {
      "agent_code": None,
      "id": "071026_674",
      "tool_needed": "no",
      "no_tool_reason": "missing_data"
    }
  },
  {
    "task_id": "071026_679",
    "result": {
      "agent_code": "plan_execute",
      "id": "071026_679",
      "tool_needed": "yes"
    }
  },
  {
    "task_id": "071026_685",
    "result": {
      "agent_code": "what_can_i_do",
      "id": "071026_685",
      "tool_needed": "yes"
    }
  },
  {
    "task_id": "071026_687",
    "result": {
      "agent_code": None,
      "id": "071026_687",
      "tool_needed": "no",
      "no_tool_reason": "clarification_needed"
    }
  },
  {
    "task_id": "071026_688",
    "result": {
      "agent_code": None,
      "id": "071026_688",
      "tool_needed": "no",
      "no_tool_reason": "clarification_needed"
    }
  },
  {
    "task_id": "071026_690",
    "result": {
      "agent_code": None,
      "id": "071026_690",
      "tool_needed": "no",
      "no_tool_reason": "no_suitable_tool"
    }
  },
  {
    "task_id": "071026_696",
    "result": {
      "agent_code": None,
      "id": "071026_696",
      "tool_needed": "no",
      "no_tool_reason": "clarification_needed"
    }
  },
  {
    "task_id": "071026_695",
    "result": {
      "agent_code": None,
      "id": "071026_695",
      "tool_needed": "no",
      "no_tool_reason": "clarification_needed"
    }
  },
  {
    "task_id": "071026_697",
    "result": {
      "agent_code": None,
      "id": "071026_697",
      "tool_needed": "no",
      "no_tool_reason": "no_suitable_tool"
    }
  },
  {
    "task_id": "071026_700",
    "result": {
      "agent_code": None,
      "id": "071026_700",
      "tool_needed": "no",
      "no_tool_reason": "clarification_needed"
    }
  },
  {
    "task_id": "071026_704",
    "result": {
      "agent_code": "what_can_i_do",
      "id": "071026_704",
      "tool_needed": "yes"
    }
  },
  {
    "task_id": "071026_705",
    "result": {
      "agent_code": None,
      "id": "071026_705",
      "tool_needed": "no",
      "no_tool_reason": "no_suitable_tool"
    }
  },
  {
    "task_id": "071026_706",
    "result": {
      "agent_code": None,
      "id": "071026_706",
      "tool_needed": "no",
      "no_tool_reason": "clarification_needed"
    }
  },
  {
    "task_id": "071026_711",
    "result": {
      "agent_code": "get_contest_methodology_tool",
      "id": "071026_711",
      "tool_needed": "yes"
    }
  },
  {
    "task_id": "071026_712",
    "result": {
      "agent_code": None,
      "id": "071026_712",
      "tool_needed": "no",
      "no_tool_reason": "no_suitable_tool"
    }
  },
  {
    "task_id": "071026_718",
    "result": {
      "agent_code": "plan_execute",
      "id": "071026_718",
      "tool_needed": "yes"
    }
  },
  {
    "task_id": "071026_723",
    "result": {
      "agent_code": "get_contest_result",
      "id": "071026_723",
      "tool_needed": "yes"
    }
  },
  {
    "task_id": "071026_734",
    "result": {
      "agent_code": "get_contest_methodology_tool",
      "id": "071026_734",
      "tool_needed": "yes"
    }
  },
  {
    "task_id": "071026_743",
    "result": {
      "agent_code": None,
      "id": "071026_743",
      "tool_needed": "no",
      "no_tool_reason": "missing_data"
    }
  },
  {
    "task_id": "071026_747",
    "result": {
      "agent_code": None,
      "id": "071026_747",
      "tool_needed": "no",
      "no_tool_reason": "unclear_request"
    }
  },
  {
    "task_id": "071026_752",
    "result": {
      "agent_code": "plan_execute",
      "id": "071026_752",
      "tool_needed": "yes"
    }
  },
  {
    "task_id": "071026_757",
    "result": {
      "agent_code": None,
      "id": "071026_757",
      "tool_needed": "no",
      "no_tool_reason": "no_suitable_tool"
    }
  },
  {
    "task_id": "071026_761",
    "result": {
      "agent_code": None,
      "id": "071026_761",
      "tool_needed": "no",
      "no_tool_reason": "no_suitable_tool"
    }
  },
  {
    "task_id": "071026_763",
    "result": {
      "agent_code": None,
      "id": "071026_763",
      "tool_needed": "no",
      "no_tool_reason": "config_request"
    }
  },
  {
    "task_id": "071026_764",
    "result": {
      "agent_code": None,
      "id": "071026_764",
      "tool_needed": "no",
      "no_tool_reason": "no_suitable_tool"
    }
  },
  {
    "task_id": "071026_769",
    "result": {
      "agent_code": None,
      "id": "071026_769",
      "tool_needed": "no",
      "no_tool_reason": "no_suitable_tool"
    }
  },
  {
    "task_id": "071026_768",
    "result": {
      "agent_code": "plan_execute",
      "id": "071026_768",
      "tool_needed": "yes"
    }
  },
  {
    "task_id": "071026_779",
    "result": {
      "agent_code": None,
      "id": "071026_779",
      "tool_needed": "no",
      "no_tool_reason": "no_suitable_tool"
    }
  },
  {
    "task_id": "071026_783",
    "result": {
      "agent_code": None,
      "id": "071026_783",
      "tool_needed": "no",
      "no_tool_reason": "no_suitable_tool"
    }
  },
  {
    "task_id": "071026_834",
    "result": {
      "agent_code": None,
      "id": "071026_834",
      "tool_needed": "no",
      "no_tool_reason": "unclear_request"
    }
  },
  {
    "task_id": "071026_836",
    "result": {
      "agent_code": None,
      "id": "071026_836",
      "tool_needed": "no",
      "no_tool_reason": "no_suitable_tool"
    }
  },
  {
    "task_id": "071026_841",
    "result": {
      "agent_code": None,
      "id": "071026_841",
      "tool_needed": "no",
      "no_tool_reason": "no_suitable_tool"
    }
  },
  {
    "task_id": "071026_849",
    "result": {
      "agent_code": "get_contest_result",
      "id": "071026_849",
      "tool_needed": "yes"
    }
  },
  {
    "task_id": "071026_786",
    "result": {
      "agent_code": "plan_execute",
      "id": "071026_786",
      "tool_needed": "yes"
    }
  },
  {
    "task_id": "071026_791",
    "result": {
      "agent_code": "get_bank_metric_tool",
      "id": "071026_791",
      "tool_needed": "yes"
    }
  },
  {
    "task_id": "071026_795",
    "result": {
      "agent_code": None,
      "id": "071026_795",
      "tool_needed": "no",
      "no_tool_reason": "no_suitable_tool"
    }
  },
  {
    "task_id": "071026_796",
    "result": {
      "agent_code": None,
      "id": "071026_796",
      "tool_needed": "no",
      "no_tool_reason": "clarification_needed"
    }
  },
  {
    "task_id": "071026_797",
    "result": {
      "agent_code": None,
      "id": "071026_797",
      "tool_needed": "no",
      "no_tool_reason": "no_suitable_tool"
    }
  },
  {
    "task_id": "071026_800",
    "result": {
      "agent_code": "get_contest_methodology_tool",
      "id": "071026_800",
      "tool_needed": "yes"
    }
  },
  {
    "task_id": "071026_804",
    "result": {
      "agent_code": "what_can_i_do",
      "id": "071026_804",
      "tool_needed": "yes"
    }
  },
  {
    "task_id": "071026_806",
    "result": {
      "agent_code": "plan_execute",
      "id": "071026_806",
      "tool_needed": "yes"
    }
  },
  {
    "task_id": "071026_814",
    "result": {
      "agent_code": "get_contest_result",
      "id": "071026_814",
      "tool_needed": "yes"
    }
  },
  {
    "task_id": "071026_822",
    "result": {
      "agent_code": None,
      "id": "071026_822",
      "tool_needed": "no",
      "no_tool_reason": "clarification_needed"
    }
  },
  {
    "task_id": "071026_825",
    "result": {
      "agent_code": None,
      "id": "071026_825",
      "tool_needed": "no",
      "no_tool_reason": "no_suitable_tool"
    }
  },
  {
    "task_id": "071026_832",
    "result": {
      "agent_code": None,
      "id": "071026_832",
      "tool_needed": "no",
      "no_tool_reason": "no_suitable_tool"
    }
  }
]

len(answers)

52

## Pool6

In [230]:
answers = [
    {
        "task_id": "071026_860",
        "result": {
            "agent_code": "get_contest_methodology_tool",
            "id": "071026_860",
            "tool_needed": "yes"
        }
    },
    {
        "task_id": "071026_865",
        "result": {
            "agent_code": "get_bank_metric_tool",
            "id": "071026_865",
            "tool_needed": "yes"
        }
    },
    {
        "task_id": "071026_872",
        "result": {
            "agent_code": "get_contest_methodology_tool",
            "id": "071026_872",
            "tool_needed": "yes"
        }
    },
    {
        "task_id": "071026_879",
        "result": {
            "agent_code": "get_contest_methodology_tool",
            "id": "071026_879",
            "tool_needed": "yes"
        }
    },
    {
        "task_id": "071026_881",
        "result": {
            "agent_code": "plan_execute",
            "id": "071026_881",
            "tool_needed": "yes"
        }
    },
    {
        "task_id": "071026_885",
        "result": {
            "agent_code": None,
            "id": "071026_885",
            "tool_needed": "no",
            "no_tool_reason": "clarification_needed"
        }
    },
    {
        "task_id": "071026_889",
        "result": {
            "agent_code": "get_contest_result",
            "id": "071026_889",
            "tool_needed": "yes"
        }
    },
    {
        "task_id": "071026_890",
        "result": {
            "agent_code": "plan_execute",
            "id": "071026_890",
            "tool_needed": "yes"
        }
    },
    {
        "task_id": "071026_897",
        "result": {
            "agent_code": None,
            "id": "071026_897",
            "tool_needed": "no",
            "no_tool_reason": "no_suitable_tool"
        }
    },
    {
        "task_id": "071026_893",
        "result": {
            "agent_code": None,
            "id": "071026_893",
            "tool_needed": "no",
            "no_tool_reason": "clarification_needed"
        }
    },
    {
        "task_id": "071026_900",
        "result": {
            "agent_code": None,
            "id": "071026_900",
            "tool_needed": "no",
            "no_tool_reason": "clarification_needed"
        }
    },
    {
        "task_id": "071026_898",
        "result": {
            "agent_code": None,
            "id": "071026_898",
            "tool_needed": "no",
            "no_tool_reason": "clarification_needed"
        }
    },
  {
    "task_id": "071026_902",
    "result": {
      "agent_code": None,
      "id": "071026_902",
      "tool_needed": "no",
      "no_tool_reason": "config_request"
    }
  },
  {
    "task_id": "071026_904",
    "result": {
      "agent_code": None,
      "id": "071026_904",
      "tool_needed": "no",
      "no_tool_reason": "clarification_needed"
    }
  },
  {
    "task_id": "071026_911",
    "result": {
      "agent_code": "get_contest_methodology_tool",
      "id": "071026_911",
      "tool_needed": "yes"
    }
  },
  {
    "task_id": "071026_915",
    "result": {
      "agent_code": "plan_execute",
      "id": "071026_915",
      "tool_needed": "yes"
    }
  },
  {
    "task_id": "071026_916",
    "result": {
      "agent_code": "get_contest_methodology_tool",
      "id": "071026_916",
      "tool_needed": "yes"
    }
  },
  {
    "task_id": "071026_925",
    "result": {
      "agent_code": "get_contest_methodology_tool",
      "id": "071026_925",
      "tool_needed": "yes"
    }
  },
  {
    "task_id": "071026_924",
    "result": {
      "agent_code": "get_bank_improvement_tool",
      "id": "071026_924",
      "tool_needed": "yes"
    }
  },
  {
    "task_id": "071026_946",
    "result": {
      "agent_code": "get_bank_metric_tool",
      "id": "071026_946",
      "tool_needed": "yes"
    }
  },
  {
    "task_id": "071026_950",
    "result": {
      "agent_code": "plan_execute",
      "id": "071026_950",
      "tool_needed": "yes"
    }
  },
  {
    "task_id": "071026_954",
    "result": {
      "agent_code": None,
      "id": "071026_954",
      "tool_needed": "no",
      "no_tool_reason": "unclear_request"
    }
  },
  {
    "task_id": "071026_966",
    "result": {
      "agent_code": None,
      "id": "071026_966",
      "tool_needed": "no",
      "no_tool_reason": "missing_data"
    }
  },
  {
    "task_id": "071026_959",
    "result": {
      "agent_code": None,
      "id": "071026_959",
      "tool_needed": "no",
      "no_tool_reason": "no_suitable_tool"
    }
  },
    {
        'task_id': '071026_971',
        'result': {
            'agent_code': 'get_contest_methodology_tool',
            'id': '071026_971',
            'tool_needed': 'yes'
        }
    },
    {
        'task_id': '071026_976',
        'result': {
            'agent_code': None,
            'id': '071026_976',
            'tool_needed': 'no',
            'no_tool_reason': 'missing_data'
        }
    },
    {
        'task_id': '071026_979',
        'result': {
            'agent_code': None,
            'id': '071026_979',
            'tool_needed': 'no',
            'no_tool_reason': 'unclear_request'
        }
    },
    {
        'task_id': '071026_982',
        'result': {
            'agent_code': None,
            'id': '071026_982',
            'tool_needed': 'no',
            'no_tool_reason': 'clarification_needed'
        }
    },
    {
        'task_id': '071026_988',
        'result': {
            'agent_code': None,
            'id': '071026_988',
            'tool_needed': 'no',
            'no_tool_reason': 'no_suitable_tool'
        }
    },
    {
        'task_id': '071026_989',
        'result': {
            'agent_code': None,
            'id': '071026_989',
            'tool_needed': 'no',
            'no_tool_reason': 'clarification_needed'
        }
    },
    {
        'task_id': '071026_998',
        'result': {
            'agent_code': 'get_kpi_rgs',
            'id': '071026_998',
            'tool_needed': 'yes'
        }
    },
    {
        'task_id': '071026_997',
        'result': {
            'agent_code': None,
            'id': '071026_997',
            'tool_needed': 'no',
            'no_tool_reason': 'clarification_needed'
        }
    }
]

len(answers)

32

## Собираем

In [172]:
# data = {
#     'task': '/Users/richardgurtsiev/Desktop/projects/vibe_code/workspaces/kib/data/вопросы/pool1.json',
#     'gt': '/Users/richardgurtsiev/Desktop/projects/vibe_code/workspaces/kib/data/ответы/pool1.json'
# }

# with open(data['task'], 'r', encoding='utf-8') as f:
#     tasks = json.load(f)
# # [
# #   {
# #     "data": {
# #       "id": "071026_50",
# #       "question": "как мне выгрузить индекс успеха в разрезе компаний и КМ",
# #       "role": "USMIMKO, KKSB, MMB, RGS",
# #       "tools": "get_bank_improvement_tool, get_bank_metric_tool, get_contest_methodology_tool, get_contest_result, get_kpi_rgs, get_rating_rgs, get_tribe, kpk_tool, plan_execute, planair_fa, trust_level_case, what_can_i_do"
# #     },
# #     "filename": "071026_50"
# #   },



# with open(data['gt'], 'r', encoding='utf-8') as f:
#     gt = json.load(f)
# # [
# #     {
# #         "assignment_id": "d4714ff3-aa5c-436a-88b9-68a757c8bffd",
# #         "end_date": "2026-10-13T15:43:21.673884+03:00",
# #         "file_name": "071026_1",
# #         "item_id": "15c25255-dd86-4733-9bc4-5fe3db141de6",
# #         "item_type": "data",
# #         "marker_id": "5415101c-fa23-4352-8412-3895912afe29",
# #         "organization_id": "ce07feb6-7405-4d32-8346-efe1eb7b08df",
# #         "person_id": "bc031b82-28c3-4172-97d7-d5cef4ec5d6f",
# #         "pool_ids": [
# #             "17d9f816-d54a-4ed1-8354-a5e3f6c29b7d"
# #         ],
# #         "price": 14.5,
# #         "result": {
# #             "id": "071026_1",
# #             "agent_code": "get_tribe",
# #             "tool_needed": "yes"
# #         },
# #         "start_date": "2026-10-13T15:41:46.659628+03:00",
# #         "status": "ACCEPTED",
# #         "task_id": "2ac23ced-f53e-4a56-89b6-15dc805ca5e3",
# #         "task_type": "PROD",
# #         "consistency": 1,
# #         "submitted_at": "2026-10-13T15:43:21.673884+03:00",
# #         "accepted_at": "2026-10-13T15:43:21.673884+03:00",
# #         "started_at": "2026-10-13T15:41:46.659628+03:00",
# #         "datetime": "2026-10-13T15:43:21.673884+03:00",
# #         "is_review_public": false
# #     },


# dataset, ties = build_majority_dataset(gt)
# print('dataset')
# print(dataset[0])
# print('\n\n')
# print('ties')
# print(ties[0])

# dataset
# {'item_id': '15c25255-dd86-4733-9bc4-5fe3db141de6', 'result': {'agent_code': 'get_tribe', 'id': '071026_1', 'tool_needed': 'yes'}, 'votes': 2, 'total_votes': 2}



# ties
# {'item_id': '445c2e31-b8fd-4e33-95ca-650e1facb3cd', 'result': [{'agent_code': 'get_kpi_rgs', 'id': '071026_5', 'tool_needed': 'yes'}, {'agent_code': None, 'id': '071026_5', 'no_tool_reason': 'clarification_needed', 'tool_needed': 'no'}, {'agent_code': 'plan_execute', 'id': '071026_5', 'tool_needed': 'yes'}], 'votes': 1, 'total_votes': 3}

In [213]:
from collections import Counter
import json


def normalize_result(result):
    """
    Приводит разметку к целевой схеме:
    - yes: agent_code + tool_needed
    - no: agent_code=None + tool_needed + no_tool_reason
    """

    tool_needed = result.get("tool_needed")

    if tool_needed == "yes":
        agent_code = result.get("agent_code")

        if not agent_code:
            raise ValueError(f"Для tool_needed='yes' не указан agent_code: {result}")

        return {
            "agent_code": agent_code,
            "tool_needed": "yes"
        }

    elif tool_needed == "no":
        no_tool_reason = result.get("no_tool_reason")

        if not no_tool_reason:
            raise ValueError(
                f"Для tool_needed='no' не указан no_tool_reason: {result}"
            )

        return {
            "agent_code": None,
            "tool_needed": "no",
            "no_tool_reason": no_tool_reason
        }

    raise ValueError(f"Неизвестное значение tool_needed: {tool_needed}")


def build_final_dataset(tasks, dataset, answers):
    # --------------------------------------------------------
    # 1. Собираем итоговые метки из результатов голосования
    # --------------------------------------------------------

    labels = {}

    for item in dataset:
        result = item["result"]
        task_id = result["id"]

        if task_id in labels:
            raise ValueError(f"Повторная метка для task_id={task_id}")

        labels[task_id] = normalize_result(result)

    # --------------------------------------------------------
    # 2. Добавляем дополнительную разметку спорных случаев
    # --------------------------------------------------------

    for answer in answers:
        task_id = answer["task_id"]
        result = normalize_result(answer["result"])

        if task_id in labels:
            if labels[task_id] != result:
                raise ValueError(
                    f"Конфликт разметки для task_id={task_id}: "
                    f"{labels[task_id]} != {result}"
                )
        else:
            labels[task_id] = result

    # --------------------------------------------------------
    # 3. Сопоставляем метки с исходными заданиями
    # --------------------------------------------------------

    final_dataset = []
    missing_labels = []

    for task in tasks:
        task_data = task["data"]
        task_id = task_data["id"]

        if task_id not in labels:
            missing_labels.append(task_id)
            continue

        final_dataset.append({
            "data": {
                "question": task_data["question"],
                "tools": task_data["tools"],
                "id": task_data["id"],
                "role": task_data["role"],
            },
            "result": labels[task_id]
        })

    return final_dataset, missing_labels

In [232]:
final_dataset, missing_labels = build_final_dataset(
    tasks=tasks,
    dataset=dataset,
    answers=answers
)

print("Размер итогового датасета:", len(final_dataset))
print("Без итоговой разметки:", len(missing_labels))

print(json.dumps(
    final_dataset[5],
    ensure_ascii=False,
    indent=4
))

Размер итогового датасета: 150
Без итоговой разметки: 0
{
    "data": {
        "question": "расходы по трайбу ДТБ по тарифным моделям за 11 месяцев 2025?",
        "tools": "get_bank_improvement_tool, get_bank_metric_tool, get_contest_methodology_tool, get_contest_result, get_tribe, kpk_tool, plan_execute, planair_fa, trust_level_case, what_can_i_do",
        "id": "071026_995",
        "role": "USMIMKO"
    },
    "result": {
        "agent_code": "get_tribe",
        "tool_needed": "yes"
    }
}


In [233]:
output_path = "./gt/pool6.json"

with open(output_path, "w", encoding="utf-8") as f:
    json.dump(
        final_dataset,
        f,
        ensure_ascii=False,
        indent=4
    )

In [163]:
# import logging
# import json
# import os
# import time
# import requests

# from pathlib import Path
# from collections import defaultdict, Counter

# from crowd_sdk.tagme import TagmeClientAdvanced


# # ============================================================
# # 1. КОНФИГУРАЦИЯ
# # ============================================================

# data = {
#     'task': './вопросы/pool1.json',
#     'gt': './ответы/pool1.json'
# }

# PROVIDER = "aihf"  # "direct", "aihf", "vsellm"

# ENDPOINTS = {
#     "direct": "https://tagme.sberdevices.ru/ai/llm/v1/chat/completions",
#     "aihf": "https://tagme.sberdevices.ru/x/aihf/llm/v1/chat/completions",
#     "vsellm": "https://tagme.sberdevices.ru/x/ai/llm/v1/chat/completions",
# }

# CFG = {
#     "direct": "~/configs/.ml_config.cfg",
#     "aihf": "~/configs/.ml_config.cfg",
#     "vsellm": "~/configs/.ml_config.cfg",
# }

# MODEL = "DeepSeek-V4.1-Flash"

# TEMPERATURE = 0.1
# MAX_TOKENS = 8000
# TIMEOUT = (60, 120)
# MAX_RETRIES = 3


# # ============================================================
# # ЛОГИРОВАНИЕ
# # ============================================================

# LOG_FILE = Path("./ответы/pool1_model_votes.log")
# LOG_FILE.parent.mkdir(parents=True, exist_ok=True)

# logger = logging.getLogger("llm_debug")
# logger.setLevel(logging.INFO)
# logger.propagate = False

# if not logger.handlers:
#     formatter = logging.Formatter(
#         "%(asctime)s | %(levelname)s | %(message)s",
#         datefmt="%Y-%m-%d %H:%M:%S"
#     )

#     file_handler = logging.FileHandler(
#         LOG_FILE,
#         encoding="utf-8"
#     )
#     file_handler.setFormatter(formatter)

#     console_handler = logging.StreamHandler()
#     console_handler.setFormatter(formatter)

#     logger.addHandler(file_handler)
#     logger.addHandler(console_handler)

# logger.info("=" * 100)
# logger.info("ЗАПУСК СКРИПТА")
# logger.info(f"Модель: {MODEL}")
# logger.info(f"Провайдер: {PROVIDER}")
# logger.info(f"Endpoint: {ENDPOINTS[PROVIDER]}")



# def build_task_context(task: dict) -> dict:
#     tools = task.get("tools", "")

#     if isinstance(tools, str):
#         available_tools = [
#             tool.strip()
#             for tool in tools.split(",")
#             if tool.strip()
#         ]
#     elif isinstance(tools, list):
#         available_tools = tools
#     else:
#         available_tools = []

#     return {
#         "id": task.get("id"),
#         "question": task.get("question"),
#         "role": task.get("role"),
#         "available_tools": available_tools,
#     }


# def build_user_prompt(
#     task: dict,
#     results: list[dict],
# ) -> str:

#     task_context = build_task_context(task)

#     options = [
#         {
#             "option": index,
#             "result": result,
#         }
#         for index, result in enumerate(results, start=1)
#     ]

#     user_prompt = f"""
# # ЗАДАЧА

# Определи, какой из предложенных вариантов result
# наиболее корректен с учётом системных инструкций.

# ## КОНТЕКСТ КАРТОЧКИ

# {json.dumps(task_context, ensure_ascii=False, indent=2)}

# ## ВАРИАНТЫ ОТВЕТОВ

# {json.dumps(options, ensure_ascii=False, indent=2)}

# ## ИНСТРУКЦИЯ

# 1. Проанализируй исходный вопрос с учётом роли пользователя.
# 2. Учитывай только инструменты из available_tools.
# 3. Сопоставь смысл запроса со справочником инструментов.
# 4. Проверь специальные ограничения и обязательные параметры.
# 5. Оцени каждый вариант result целиком.
# 6. Выбери один наиболее корректный вариант.
# 7. Не объединяй поля разных вариантов.
# 8. Верни ответ строго в JSON по формату системного промпта.

# Если ни один вариант нельзя обоснованно выбрать из-за недостатка
# контекста, используй decision = insufficient_context.
# """

#     return user_prompt.strip()


# # ============================================================
# # 2. КЛИЕНТ ДЛЯ ОБРАЩЕНИЯ К МОДЕЛИ
# # ============================================================

# class GigaChatClient:

#     def __init__(self):
#         self.url = ENDPOINTS[PROVIDER]
#         self.model = MODEL
#         self.token = self._get_token()

#         self.session = requests.Session()

#     @staticmethod
#     def _get_token() -> str:
#         cfg_path = os.path.expanduser(CFG[PROVIDER])

#         client = TagmeClientAdvanced(cfg_path)

#         return client.client.token

#     def invoke(
#         self,
#         system_prompt: str,
#         user_prompt: str,
#         task_id: str = None,
#         item_id: str = None,
#     ) -> str:

#         payload = {
#             "model": self.model,
#             "messages": [
#                 {
#                     "role": "system",
#                     "content": system_prompt,
#                 },
#                 {
#                     "role": "user",
#                     "content": user_prompt,
#                 },
#             ],
#             "temperature": TEMPERATURE,
#             "max_tokens": MAX_TOKENS,
#             "profanity_check": False,
#         }

#         headers = {
#             "Authorization": f"Bearer {self.token}",
#             "Content-Type": "application/json",
#         }

#         request_info = (
#             f"task_id={task_id} | item_id={item_id}"
#         )

#         # --------------------------------------------------------
#         # ЛОГИРУЕМ ФАКТИЧЕСКИЙ ВХОД МОДЕЛИ
#         # --------------------------------------------------------

#         logger.info("\n" + "=" * 100)
#         logger.info(f"ОТПРАВКА ЗАПРОСА | {request_info}")
#         logger.info("=" * 100)

#         logger.info(
#             "ПАРАМЕТРЫ ЗАПРОСА:\n%s",
#             json.dumps(
#                 {
#                     "model": payload["model"],
#                     "temperature": payload["temperature"],
#                     "max_tokens": payload["max_tokens"],
#                     "profanity_check": payload["profanity_check"],
#                 },
#                 ensure_ascii=False,
#                 indent=4
#             )
#         )

#         logger.info(
#             "\nSYSTEM PROMPT:\n%s",
#             system_prompt
#         )

#         logger.info(
#             "\nUSER PROMPT:\n%s",
#             user_prompt
#         )

#         # Логируем полный payload без HTTP-заголовков и токена.
#         logger.info(
#             "\nПОЛНЫЙ PAYLOAD:\n%s",
#             json.dumps(
#                 payload,
#                 ensure_ascii=False,
#                 indent=4
#             )
#         )

#         token_refreshed = False

#         for attempt in range(1, MAX_RETRIES + 1):

#             try:

#                 start_time = time.perf_counter()

#                 logger.info(
#                     f"HTTP REQUEST | {request_info} | "
#                     f"Попытка {attempt}/{MAX_RETRIES}"
#                 )

#                 response = self.session.post(
#                     self.url,
#                     headers=headers,
#                     json=payload,
#                     verify=False,
#                     timeout=TIMEOUT,
#                 )

#                 elapsed_time = time.perf_counter() - start_time

#                 # ------------------------------------------------
#                 # ЛОГИРУЕМ HTTP-ОТВЕТ
#                 # ------------------------------------------------

#                 logger.info(
#                     f"HTTP RESPONSE | {request_info} | "
#                     f"Status: {response.status_code} | "
#                     f"Time: {elapsed_time:.2f} sec"
#                 )

#                 logger.info(
#                     "\nСЫРОЙ ОТВЕТ API:\n%s",
#                     response.text
#                 )

#                 # ------------------------------------------------
#                 # ОБНОВЛЕНИЕ ТОКЕНА
#                 # ------------------------------------------------

#                 if response.status_code == 401 and not token_refreshed:

#                     logger.warning(
#                         f"401 Unauthorized | {request_info} | "
#                         "Обновляем токен"
#                     )

#                     self.token = self._get_token()
#                     headers["Authorization"] = f"Bearer {self.token}"

#                     token_refreshed = True
#                     continue

#                 # ------------------------------------------------
#                 # ПОВТОР ПРИ ВРЕМЕННЫХ ОШИБКАХ
#                 # ------------------------------------------------

#                 if response.status_code == 429 or response.status_code >= 500:

#                     if attempt < MAX_RETRIES:

#                         sleep_time = 2 ** attempt

#                         logger.warning(
#                             f"Временная ошибка API: {response.status_code} | "
#                             f"Повтор через {sleep_time} сек."
#                         )

#                         time.sleep(sleep_time)
#                         continue

#                 response.raise_for_status()

#                 data = response.json()

#                 choices = data.get("choices") or []

#                 if not choices:
#                     raise ValueError(
#                         f"Модель вернула пустой choices: {data}"
#                     )

#                 content = (
#                     choices[0].get("message") or {}
#                 ).get("content", "")

#                 if isinstance(content, list):
#                     content = " ".join(
#                         part.get("text", "")
#                         for part in content
#                         if isinstance(part, dict)
#                     )

#                 if not isinstance(content, str) or not content.strip():
#                     raise ValueError(
#                         f"Модель вернула пустой ответ: {data}"
#                     )

#                 # ------------------------------------------------
#                 # ЛОГИРУЕМ ФИНАЛЬНЫЙ ОТВЕТ МОДЕЛИ
#                 # ------------------------------------------------

#                 logger.info("\n" + "=" * 100)
#                 logger.info(f"ОТВЕТ МОДЕЛИ | {request_info}")
#                 logger.info("=" * 100)

#                 logger.info(
#                     "\nИЗВЛЕЧЁННЫЙ CONTENT:\n%s",
#                     content
#                 )

#                 logger.info(
#                     f"Время выполнения: {elapsed_time:.2f} сек."
#                 )

#                 logger.info("=" * 100 + "\n")

#                 return content

#             except requests.RequestException:

#                 logger.exception(
#                     f"HTTP ERROR | {request_info} | "
#                     f"Попытка {attempt}/{MAX_RETRIES}"
#                 )

#                 if attempt == MAX_RETRIES:
#                     raise

#                 sleep_time = 2 ** attempt

#                 logger.info(
#                     f"Повтор запроса через {sleep_time} сек."
#                 )

#                 time.sleep(sleep_time)

#         raise RuntimeError(
#             "Не удалось получить ответ от модели"
#         )


# # ============================================================
# # 3. ЧТЕНИЕ ЗАДАНИЙ И ОТВЕТОВ
# # ============================================================

# with open(data['task'], 'r', encoding='utf-8') as f:
#     tasks = json.load(f)

# with open(data['gt'], 'r', encoding='utf-8') as f:
#     gt = json.load(f)


# # ============================================================
# # 4. ФОРМИРОВАНИЕ DATASET И TIES
# # ============================================================

# def build_majority_dataset(
#     gt: list[dict]
# ) -> tuple[list[dict], list[dict]]:

#     grouped = defaultdict(list)

#     for item in gt:
#         item_id = item.get('item_id')
#         result = item.get('result')

#         if item_id is None or not isinstance(result, dict):
#             continue

#         grouped[item_id].append(result)

#     dataset = []
#     ties = []

#     for item_id, results in grouped.items():

#         result_counts = Counter(
#             json.dumps(
#                 result,
#                 sort_keys=True,
#                 ensure_ascii=False
#             )
#             for result in results
#         )

#         max_votes = max(result_counts.values())

#         winners = [
#             json.loads(result)
#             for result, count in result_counts.items()
#             if count == max_votes
#         ]

#         record = {
#             'item_id': item_id,
#             'result': winners[0] if len(winners) == 1 else winners,
#             'votes': max_votes,
#             'total_votes': len(results),
#         }

#         if len(winners) == 1:
#             dataset.append(record)
#         else:
#             ties.append(record)

#     return dataset, ties


# dataset, ties = build_majority_dataset(gt)


# # ============================================================
# # 5. ИНДЕКС ЗАДАНИЙ
# # ============================================================

# tasks_by_id = {
#     task.get('data', {}).get('id'): task['data']
#     for task in tasks
#     if task.get('data', {}).get('id') is not None
# }


# def get_task_by_id(
#     tasks_by_id: dict,
#     task_id: str
# ) -> dict | None:

#     return tasks_by_id.get(task_id)


# # ============================================================
# # 6. SYSTEM PROMPT
# # ============================================================

# SYSTEM_PROMPT = """
# # СИСТЕМНЫЙ ПРОМПТ — ЭКСПЕРТНАЯ ОЦЕНКА РАЗМЕТКИ ВЫБОРА ИНСТРУМЕНТОВ АГЕНТА АНАЛИТИКА КИБ

# ## 1. РОЛЬ И ЦЕЛЬ

# Ты — эксперт по оценке корректности маршрутизации запросов в ИИ-агенте «Помощник аналитика КИБ» Сбербанка.

# Твоя специализация — оценка решений о необходимости вызова инструмента и правильности выбора инструмента для первого ответа агентa.

# Твоя задача — разрешать разногласия между несколькими независимыми разметчиками.

# На вход ты получаешь:
# - карточку задания: question, role, available_tools;
# - несколько вариантов ответов, предложенных разметчиками.

# Необходимо определить, какой из предложенных вариантов наиболее соответствует правилам работы агента и инструкции по разметке.

# **Ты оцениваешь корректность разметки, а не отвечаешь на запрос пользователя.**

# Не пытайся самостоятельно решать пользовательскую задачу, получать фактические данные или имитировать результат выполнения инструмента.

# ## 2. ОБЪЕКТ ОЦЕНКИ

# Каждый вариант ответа содержит объект result.

# Основные поля:
# - id — идентификатор задания;
# - agent_code — выбранный инструмент или соответствующее значение разметки;
# - tool_needed — необходимость вызова инструмента.

# Оценивай весь объект result целиком.

# Особенно важно:
# - tool_needed определяет, нужен ли вызов инструмента в первом ответе;
# - agent_code определяет выбранный инструмент или соответствующее значение при отсутствии вызова.

# Не разделяй эти поля при оценивании. Правильность определяется совместной комбинацией значений.

# Если два варианта совпадают по одному полю, но отличаются по другому, это разные варианты ответа.

# ## 3. ИЕРАРХИЯ ИСТОЧНИКОВ ПРАВИЛ

# При оценивании используй следующие источники в указанном порядке:

# 1. Инструкция по оценке разметки — основные правила определения правильного решения.
# 2. Справочник инструментов — назначение, ограничения, обязательные параметры и правила маршрутизации.
# 3. Системный промпт агента — логика поведения, порядок действий, правила уточнения и ограничения.
# 4. Разобранные примеры — дополнительные иллюстрации применения правил.

# Не интерпретируй примеры как исчерпывающий перечень ситуаций.

# Если общее правило и специальное правило инструмента противоречат друг другу, учитывай более конкретное специальное правило, если оно не противоречит инструкции разметки.

# ## 4. ЗНАНИЯ ОБ АГЕНТЕ

# Ты должен учитывать, что агент — помощник аналитика КИБ.

# Его основные направления работы:

# - КПЭ и выполнение планов по показателям;
# - данные по КПК и ГОСБ;
# - рейтинги РГС;
# - Единое конкурсное пространство КИБ;
# - трайбы;
# - уровень доверия клиентов;
# - внутренние методики банка;
# - анализ причин выполнения и невыполнения планов;
# - рекомендации по улучшению позиций в рейтингах.

# Агент работает только в рамках доступных ему инструментов.

# Наличие подходящего инструмента в общем справочнике не означает, что его можно использовать в конкретной карточке. Обязательно проверяй available_tools.

# ### 4.1. Правила доступности инструментов

# В карточке available_tools перечислены инструменты, доступные пользователю с учётом всех его ролей.

# Если указано несколько ролей, их права объединяются.

# Выбирать можно только инструмент из available_tools.

# Запрещено:
# - выбирать инструмент, отсутствующий в available_tools;
# - предлагать альтернативный инструмент, которого нет в списке;
# - считать, что наличие инструмента в справочнике автоматически означает наличие доступа к нему.

# ### 4.2. Справочник инструментов

# Используй полный справочник инструментов, приведённый ниже в системном промпте.

# Для каждого инструмента учитывай:

# - назначение;
# - типы запросов, для которых он предназначен;
# - специальные условия вызова;
# - обязательные и необязательные параметры;
# - ограничения;
# - исключения;
# - приоритетные правила маршрутизации.

# Не определяй инструмент исключительно по совпадению ключевых слов. Сравнивай смысл запроса с полным описанием инструмента.

# ## 5. ГЛАВНЫЙ ПРИНЦИП ОЦЕНИВАНИЯ

# Определяй, какое действие должен совершить агент в ПЕРВОМ ответе на запрос пользователя.

# Возможны два решения:

# ### 5.1. tool_needed = yes

# Выбирай этот вариант, если:

# - запрос соответствует назначению доступного инструмента;
# - инструмент можно вызвать непосредственно сейчас;
# - отсутствуют обязательные данные, которые необходимо предварительно запросить;
# - нет неоднозначности, требующей выбора между разными источниками;
# - специальное правило не запрещает вызов.

# Если запрос включает несколько действий, определяй инструмент, который должен быть вызван первым.

# ### 5.2. tool_needed = no

# Выбирай этот вариант, если первым действием агента должно быть что-либо из следующего:

# - уточнение обязательных данных;
# - уточнение смысла запроса;
# - выбор между несколькими источниками;
# - сообщение об отсутствии подходящего инструмента;
# - отказ или перенаправление согласно правилам агента;
# - ответ без использования инструмента;
# - обработка запроса служебной конфигурации.

# Не считай отсутствие всех параметров автоматической причиной отказа от вызова. Проверяй описание конкретного инструмента: многие параметры необязательны.

# ## 6. АЛГОРИТМ ОЦЕНКИ

# Для каждого задания выполняй следующие шаги строго по порядку.

# ### Шаг 1. Анализ пользовательского запроса

# Определи:
# - основную цель пользователя;
# - какие данные или действия ему необходимы;
# - есть ли несколько возможных трактовок;
# - требуется ли фактическая информация из внешних источников.

# Не додумывай отсутствующий контекст.

# ### Шаг 2. Анализ доступных инструментов

# Изучи available_tools.

# Сопоставь запрос с описаниями подходящих инструментов в справочнике.

# Учитывай:
# - назначение инструмента;
# - ограничения;
# - специальные правила;
# - обязательные параметры;
# - правила выбора между похожими инструментами.

# ### Шаг 3. Определение первого действия агента

# Ответь на вопрос:

# Может ли агент вызвать подходящий инструмент прямо сейчас или сначала должен получить дополнительную информацию от пользователя?

# Если требуется уточнение, инструмент в первом ответе не вызывается.

# ### Шаг 4. Проверка каждого варианта разметки

# Для каждого предложенного варианта независимо оцени:

# 1. Правильно ли определена необходимость вызова инструмента?
# 2. Если вызов не нужен, соответствует ли выбранное значение agent_code причине отказа или уточнения?
# 3. Если вызов нужен, соответствует ли выбранный инструмент запросу?
# 4. Присутствует ли инструмент в available_tools?
# 5. Не нарушает ли решение специальные правила инструмента?
# 6. Не пропускает ли вариант обязательное уточнение?

# ### Шаг 5. Сравнение вариантов

# Сравни все варианты по одинаковым критериям.

# Не используй следующие признаки как основание для выбора:
# - порядок появления ответа;
# - количество символов в ответе;
# - субъективная убедительность формулировки;
# - количество голосов первоначальных разметчиков;
# - совпадение с первым предложенным вариантом.

# Выбирай вариант, который наиболее полно соответствует правилам.

# ## 7. ПРАВИЛА РАЗРЕШЕНИЯ СПОРНЫХ СЛУЧАЕВ

# ### 7.1. Не хватает части данных

# Проверяй описание инструмента.

# Если параметр необязательный или его разрешено не передавать — вызов необходим.

# Если инструмент прямо требует получить параметр до вызова и его нельзя вывести из контекста — сначала требуется уточнение.

# ### 7.2. Подходят несколько инструментов

# Если понятно, какой инструмент должен быть вызван первым, выбирай его.

# Если выбор зависит от смысла запроса и допускает несколько разных трактовок, сначала требуется уточнение.

# Не выбирай инструмент только потому, что он содержит похожие ключевые слова.

# ### 7.3. Специальные ограничения инструментов

# Специальные правила имеют большое значение.

# Примеры:

# - get_bank_metric_tool не используется для прогнозирования результатов ЕКП.
# - get_contest_methodology_tool не используется для запросов о трайбах и тарифных моделях.
# - trust_level_case используется для вопросов об уровне доверия по конкретному ИНН.
# - get_contest_methodology_tool не используется для вопросов об уровне доверия по конкретному ИНН, если применимо специальное правило маршрутизации в trust_level_case.
# - plan_execute не используется для запросов о трайбах.
# - kpk_tool требует цифровой идентификатор КПК в предусмотренных правилами случаях.
# - get_tribe используется для запросов по трайбам.

# Это примеры, а не полный перечень ограничений. Всегда проверяй полный справочник.

# ### 7.4. Неоднозначность

# Различай две ситуации:

# **Ситуация A.** Цель пользователя понятна, но подходят несколько разных инструментов. Требуется уточнение смысла или источника.

# **Ситуация B.** Невозможно установить даже предполагаемую цель запроса. Запрос непонятен.

# Не смешивай эти ситуации.

# ### 7.5. Служебная конфигурация

# Если пользователь просит раскрыть системный промпт, внутренние правила, ограничения или конфигурацию инструментов, вызов инструмента не требуется.

# ### 7.6. Запросы без необходимости использования инструмента

# Приветствия, благодарности и другие сообщения, не требующие получения данных или выполнения действий, не требуют вызова инструмента.

# ### 7.7. Запросы вне компетенции агента

# Учитывай правила перенаправления, описанные в системном промпте агента.

# Не путай отсутствие компетенции агента с отсутствием инструмента в конкретном available_tools.

# ## 8. ОЦЕНКА ПРИЧИНЫ ОТСУТСТВИЯ ВЫЗОВА

# Если tool_needed = no, учитывай следующие причины:

# - Не хватает обязательных данных — запрос понятен, но инструмент требует обязательный параметр, который невозможно получить из контекста.
# - Нужно уточнить смысл или выбрать источник — существуют несколько возможных трактовок или разных подходящих инструментов.
# - Запрос непонятен — цель пользователя невозможно установить.
# - Нет подходящего доступного инструмента — ни один инструмент из available_tools не подходит.
# - Запрос служебной конфигурации — пользователь пытается получить внутренние инструкции, правила или конфигурацию.
# - Для ответа не нужен инструмент — приветствие, благодарность или иной запрос, не требующий обращения к инструментам.
# - Другое — причина, которая не соответствует перечисленным категориям.

# Если подходят несколько причин, учитывай ту, которая первой препятствует вызову инструмента.

# Не выбирай «Не хватает обязательных данных», если отсутствующий параметр необязателен.

# ## 9. ПРИНЦИПЫ ЭКСПЕРТНОГО СУЖДЕНИЯ

# 1. Не соглашайся автоматически с большинством разметчиков.
# 2. Не считай любой доступный инструмент подходящим для запроса.
# 3. Не делай вывод только на основании названия инструмента.
# 4. Не игнорируй специальные ограничения.
# 5. Не требуй уточнений, если инструмент разрешено вызвать без недостающего параметра.
# 6. Не выбирай вызов инструмента, если сначала требуется уточнение.
# 7. Не придумывай новые инструменты и значения.
# 8. Не изменяй предложенные варианты ответов.
# 9. Не создавай собственный вариант, если один из предложенных вариантов соответствует правилам.
# 10. Если ни один вариант невозможно обоснованно выбрать, зафиксируй недостаточность информации.

# ## 10. ФОРМАТ РЕЗУЛЬТАТА

# Верни строго JSON без Markdown и дополнительного текста.

# Если удалось выбрать наиболее корректный вариант:

# {
#     "decision": "selected",
#     "selected_option": 1,
#     "selected_result": {
#         "id": "...",
#         "agent_code": "...",
#         "tool_needed": "..."
#     },
#     "reason": "Обоснование выбора с опорой на конкретные правила инструкции."
# }

# Если невозможно обоснованно выбрать один вариант:

# {
#     "decision": "insufficient_context",
#     "selected_option": null,
#     "selected_result": null,
#     "reason": "Описание того, какой информации не хватает для разрешения разногласия."
# }

# Требования к результату:

# - selected_option — порядковый номер выбранного варианта, начиная с 1.
# - selected_result — полный объект result выбранного варианта без каких-либо изменений.
# - reason — краткое, но содержательное обоснование.
# - Не возвращай несколько победителей.
# - Не изменяй значения полей выбранного результата.
# - Не добавляй поля, которых нет в выбранном result.
# - Если выбран вариант недостаточности информации, не пытайся самостоятельно сформировать новый эталонный ответ.

# ## 11. КРИТЕРИЙ КАЧЕСТВА

# Главная цель — определить, какое решение действительно должен принять агент аналитика КИБ в первом ответе, исходя из запроса пользователя, доступных инструментов и всех применимых правил.

# Правильный ответ — не тот, который выглядит наиболее правдоподобно, а тот, который лучше всего подтверждается инструкцией и описаниями инструментов.
# """.strip()





# # ============================================================
# # 7. USER PROMPT
# # ============================================================

# USER_PROMPT = """
# # КАРТОЧКА ЗАДАНИЯ

# {task}

# # ВАРИАНТЫ ОТВЕТОВ РАЗМЕТЧИКОВ

# {ground_truth_str}

# Проанализируй карточку и все предложенные варианты.
# Выбери наиболее корректный вариант согласно системной инструкции.
# """.strip()

# # ============================================================
# # 8. ПОДГОТОВКА PROMPTS
# # ============================================================

# # ============================================================
# # 8. ПОДГОТОВКА PROMPTS
# # ============================================================

# prompts_task = []

# logger.info("=" * 100)
# logger.info("ПОДГОТОВКА ЗАПРОСОВ")
# logger.info(f"Всего заданий с ничьими: {len(ties)}")

# for tie in ties:

#     results = tie['result']

#     # Проверяем наличие вариантов
#     if not results:
#         logger.warning(
#             f"Нет вариантов ответов: {tie['item_id']}"
#         )
#         continue

#     # Получаем ID задания
#     task_ids = {
#         result.get("id")
#         for result in results
#     }

#     if None in task_ids or len(task_ids) != 1:
#         logger.warning(
#             f"Некорректные ID вариантов: "
#             f"item_id={tie['item_id']}, ids={task_ids}"
#         )
#         continue

#     task_id = next(iter(task_ids))

#     # Находим исходное задание
#     task = get_task_by_id(tasks_by_id, task_id)

#     if task is None:
#         logger.warning(
#             f"Задание не найдено: {task_id}"
#         )
#         continue

#     # Формируем user prompt через нормализованный контекст
#     user_prompt = build_user_prompt(
#         task=task,
#         results=results,
#     )

#     prompts_task.append({
#         'task_id': task_id,
#         'item_id': tie['item_id'],
#         'results': results,
#         'system_prompt': SYSTEM_PROMPT,
#         'user_prompt': user_prompt,
#     })

# logger.info(
#     f"Подготовлено запросов к модели: {len(prompts_task)}"
# )
# logger.info("=" * 100)


# # ============================================================
# # 9. ИНИЦИАЛИЗАЦИЯ МОДЕЛИ
# # ============================================================

# llm = GigaChatClient()


# # ============================================================
# # 10. ОТПРАВКА ЗАПРОСОВ К МОДЕЛИ
# # ============================================================

# model_results = []

# output_dir = Path(data['gt']).parent
# output_file = output_dir / 'pool1_model_votes.json'

# for idx, item in enumerate(prompts_task, start=1):

#     print(
#         f"\n[{idx}/{len(prompts_task)}] "
#         f"Обрабатываем task_id: {item['task_id']}"
#     )

#     try:

#         response = llm.invoke(
#             system_prompt=item['system_prompt'],
#             user_prompt=item['user_prompt'],
#             task_id=item['task_id'],
#             item_id=item['item_id'],
#         )

#         record = {
#             'task_id': item['task_id'],
#             'item_id': item['item_id'],
#             'status': 'success',
#             'results': item['results'],
#             'model': MODEL,
#             'response': response,
#         }

#         print("Ответ модели получен.")

#     except Exception as e:

#         logger.exception(
#             f"ОШИБКА ОБРАБОТКИ | "
#             f"task_id={item['task_id']} | "
#             f"item_id={item['item_id']}"
#         )

#         record = {
#             'task_id': item['task_id'],
#             'item_id': item['item_id'],
#             'status': 'error',
#             'results': item['results'],
#             'model': MODEL,
#             'error': str(e),
#         }


#         print(f"Ошибка: {e}")

#     model_results.append(record)

#     # Сохраняем результаты после каждого запроса,
#     # чтобы не потерять уже полученные ответы.
#     with open(output_file, 'w', encoding='utf-8') as f:
#         json.dump(
#             model_results,
#             f,
#             ensure_ascii=False,
#             indent=4
#         )
#     logger.info(
#     f"Результат сохранён: {output_file}"
# )